# Fraud Detection Model Testing

This notebook tests the accuracy, fairness, and robustness of the trained fraud detection models.

In [40]:
import pandas as pd
import numpy as np
import onnxruntime as rt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import os

%matplotlib inline

## 1. Data Loading and Preprocessing
We load the dataset and apply the same preprocessing steps as used during training.

In [41]:
# Load the dataset
data_path = '../data/investigation_train_large_checked.csv'
if not os.path.exists(data_path):
    # Fallback for different CWD
    data_path = 'data/investigation_train_large_checked.csv'

try:
    df = pd.read_csv(data_path)
    print(f"Loaded dataset with shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: Dataset not found at {data_path}")

# Preprocessing
# Drop 'Ja' and 'Nee' columns if they exist
cols_to_drop = [col for col in ['Ja', 'Nee'] if col in df.columns]
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped columns: {cols_to_drop}")

# Convert 'checked' target column to integer (0/1)
if df['checked'].dtype == 'bool' or df['checked'].dtype == 'object':
    df['checked'] = df['checked'].astype(int)

print("Target distribution:")
print(df['checked'].value_counts())

# Split into X and y
X = df.drop(columns=['checked'])
y = df['checked']

# Create a test set
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Test set shape: {X_test.shape}")

Loaded dataset with shape: (130000, 318)
Dropped columns: ['Ja', 'Nee']
Target distribution:
checked
0    110496
1     19504
Name: count, dtype: int64
Test set shape: (26000, 315)


## 2. Model Loading and Inference Helpers
Functions to load ONNX models and generate predictions.

In [42]:
def load_model(model_path):
    """Loads an ONNX model session."""
    try:
        session = rt.InferenceSession(model_path)
        print(f"Successfully loaded model from {model_path}")
        return session
    except Exception as e:
        print(f"Error loading model {model_path}: {e}")
        return None

def predict_onnx(session, X_input):
    """
    Generates predictions using an ONNX session.
    Assumes all required fields exist in X_input.
    """
    input_meta = session.get_inputs()
    inputs = {}
    
    for meta in input_meta:
        name = meta.name
        # If the model expects specific features by name, we extract them.
        if name in X_input.columns:
            inputs[name] = X_input[name].values.astype(np.float32).reshape(-1, 1)
        elif len(input_meta) == 1:
            # Fallback for single matrix input
            inputs[name] = X_input.to_numpy(dtype=np.float32)

    output_names = [meta.name for meta in session.get_outputs()]
    try:
        preds = session.run(output_names, inputs)
        return preds[0]
    except Exception as e:
        print(f"Prediction error: {e}")
        return None

## 3. Standard Evaluation
We define metrics and the specific feature sets for each model.
Model 1 (Good) excludes proxy variables.
Model 2 (Bad) includes them.

In [43]:
# Define proxy variables (sensitive attributes)
proxy_vars = [
    # Neighbourhood
    "adres_recentste_buurt_groot_ijsselmonde",
    "adres_recentste_buurt_nieuwe_westen",
    "adres_recentste_buurt_other",
    "adres_recentste_buurt_oude_noorden",
    "adres_recentste_buurt_vreewijk",
    "adres_aantal_verschillende_wijken",

    # Language-related proxies
    "belemmering_hist_taal",
    "contacten_onderwerp_beoordelen_taaleis",
    "contacten_onderwerp_boolean_beoordelen_taaleis",
    "contacten_onderwerp_boolean_taaleis___voldoet",
    "persoonlijke_eigenschappen_dagen_sinds_taaleis",
    "persoonlijke_eigenschappen_spreektaal",
    "persoonlijke_eigenschappen_spreektaal_anders",
    "persoonlijke_eigenschappen_taaleis_schrijfv_ok",
    "persoonlijke_eigenschappen_taaleis_voldaan"
]

def evaluate_model(session, X_test_all, y_test):
    """
    Evaluates the model on the test set.
    """
    print(f"--- Evaluating Model ---")
    
    # Predict (predict_onnx handles feature selection)
    y_pred = predict_onnx(session, X_test_all)
    
    if y_pred is None:
        print("Skipping evaluation due to prediction failure.")
        return
        
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    return y_pred

## 4. Fairness Testing (Partitioning)
We partition the test data based on sensitive attributes (e.g., language barrier) and compare the False Positive Rate (FPR) between groups.
- **Hypothesis**: A fair model should have similar FPR across groups. A biased model might flag one group more often incorrectly.

In [ ]:
def calculate_fpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    if (fp + tn) == 0:
        return 0.0
    return fp / (fp + tn)

def test_partitioning(session, X_test_all, y_test):
    """
    Tests fairness by partitioning data on proxy variables.
    """
    print(f"\n--- Fairness Test (Partitioning) ---")
    
    # Predict (predict_onnx handles feature selection)
    y_pred_all = predict_onnx(session, X_test_all)
    if y_pred_all is None:
        return

    # Create a dataframe for easy slicing
    results_df = X_test_all.copy()
    results_df['target'] = y_test.values
    results_df['prediction'] = y_pred_all
    
    for proxy in proxy_vars:
        if proxy not in results_df.columns:
            continue
            
        print(f"\nPartitioning by: {proxy}")
        
        # Group 1: Has the attribute (e.g., language barrier = 1)
        group_1 = results_df[results_df[proxy] == 1]
        # Group 0: Does not have it
        group_0 = results_df[results_df[proxy] == 0]
        
        if len(group_1) == 0 or len(group_0) == 0:
            print("  Skipping: One partition is empty.")
            continue
            
        fpr_1 = calculate_fpr(group_1['target'], group_1['prediction'])
        fpr_0 = calculate_fpr(group_0['target'], group_0['prediction'])
        
        count_1 = len(group_1)
        count_0 = len(group_0)
        
        print(f"  Group 1 (Has feature, n={count_1}): FPR = {fpr_1:.4f}")
        print(f"  Group 0 (No feature,  n={count_0}): FPR = {fpr_0:.4f}")
        print(f"  Diff (G1 - G0): {fpr_1 - fpr_0:.4f}")
        
        if abs(fpr_1 - fpr_0) > 0.05:
            print("\033[91m  (!) Significant disparity detected (> 5%)\033[0m")
        else:
            print("\033[92m  (OK) Disparity within 5%\033[0m")

## 5. Metamorphic Testing (Robustness)
We apply a metamorphic transformation: **Flipping a sensitive attribute**.
- **Relation**: The prediction should remain **invariant** (unchanged) if the model is fair and does not use the attribute.
- **Test**:
    1. Take a sample of users.
    2. Predict.
    3. Flip `belemmering_hist_taal` (Language Barrier) (0->1, 1->0).
    4. Predict again.
    5. Assert `Prediction 1 == Prediction 2`.

In [ ]:
def test_metamorphic(session, X_test_all, sample_size=1000):
    """
    Tests robustness by flipping sensitive features defined in proxy_vars.
    """
    print(f"\n--- Metamorphic Test (Robustness) ---")
    
    # Select a sample to speed up testing
    sample = X_test_all.sample(n=min(sample_size, len(X_test_all)), random_state=42).copy()
    
    # Original prediction
    preds_orig = predict_onnx(session, sample)
    if preds_orig is None:
        print("Original predictions failed.")
        return

    robustness_scores = {}

    for feature in proxy_vars:
        if feature not in sample.columns:
            # print(f"Skipping {feature}: Not found in columns.")
            continue
            
        # Determine transformation strategy
        # Check if binary
        unique_vals = np.unique(sample[feature].dropna())
        is_binary = np.all(np.isin(unique_vals, [0, 1]))
        
        sample_flipped = sample.copy()
        
        if is_binary:
            # Flip 0->1 and 1->0
            sample_flipped[feature] = 1 - sample_flipped[feature]
            transform_desc = "flipped"
        else:
            # For non-binary, we permute the values to ensure valid domain values
            # but break the correlation with the target for this specific instance.
            # This tests if the model is relying on the specific value of this feature.
            sample_flipped[feature] = np.random.permutation(sample_flipped[feature].values)
            transform_desc = "permuted"

        # New prediction
        preds_new = predict_onnx(session, sample_flipped)
        
        if preds_new is None:
            continue
            
        # Compare
        changes = np.sum(preds_orig != preds_new)
        total = len(sample)
        consistency = (total - changes) / total
        
        robustness_scores[feature] = consistency
        
        print(f"Feature: {feature:50} | Transform: {transform_desc:10} | Score: {consistency:.4f}")
        
        if consistency < 1.0:
            print(f"\033[91m    -> FAIL: {changes} changes detected.\033[0m")
    
    # Summary
    print("\nRobustness Summary:")
    pass_count = sum(1 for s in robustness_scores.values() if s == 1.0)
    print(f"Passed: {pass_count}/{len(robustness_scores)} features.")
    if pass_count < len(robustness_scores):
        print("\033[91mSome sensitive features affect the model predictions!\033[0m")
    else:
        print("\033[92mModel is robust to all sensitive features tested.\033[0m")

## 6. Execution
Run the full test suite for the selected model.

In [ ]:
# Model Selection
# Change this path to test a different model
MODEL_PATH = "model_1.onnx" 
# MODEL_PATH = "model_2.onnx"

print(f"Loading model from: {MODEL_PATH}")
session = load_model(MODEL_PATH)

if session:
    evaluate_model(session, X_test, y_test)
    test_partitioning(session, X_test, y_test)
    test_metamorphic(session, X_test)
else:
    print("Model loading failed.")

Loading model from: model_1.onnx
Successfully loaded model from model_1.onnx
--- Evaluating Model ---
Accuracy:  0.9478
Precision: 0.8423
Recall:    0.8026
F1 Score:  0.8220

Confusion Matrix:
[[21513   586]
 [  770  3131]]

--- Fairness Test (Partitioning) ---

Partitioning by: adres_recentste_buurt_groot_ijsselmonde
  Group 1 (Has feature, n=104): FPR = 0.0345
  Group 0 (No feature,  n=25896): FPR = 0.0265
  Diff (G1 - G0): 0.0080
  (OK) Disparity within 5%

Partitioning by: adres_recentste_buurt_nieuwe_westen
  Group 1 (Has feature, n=81): FPR = 0.0725
  Group 0 (No feature,  n=25919): FPR = 0.0264
  Diff (G1 - G0): 0.0461
  (OK) Disparity within 5%

Partitioning by: adres_recentste_buurt_other
  Group 1 (Has feature, n=12838): FPR = 0.0229
  Group 0 (No feature,  n=13162): FPR = 0.0301
  Diff (G1 - G0): -0.0072
  (OK) Disparity within 5%

Partitioning by: adres_recentste_buurt_oude_noorden
  Group 1 (Has feature, n=31): FPR = 0.0357
  Group 0 (No feature,  n=25969): FPR = 0.0265
  